## **Income Prediction Using Random Forest Classifier with Hyperparameter Tuning**

### **Problem Statement**

The income classification problem involves predicting whether an individual 
earns more or less than $50K annually based on demographic and employment 
attributes such as age, education, occupation, marital status, and work hours.

A single Decision Tree often overfits and lacks generalization. This project 
addresses that limitation by using Random Forest — a powerful ensemble method 
that combines multiple Decision Trees to deliver more accurate and stable 
predictions on the Adult Census Dataset (32,561 records).



### **Objective**
- Predict income category (<=50K or >50K) using Random Forest Classifier

- Handle missing values encoded as ' ?' in the Adult dataset
- Apply One-Hot Encoding with drop_first=True to avoid multicollinearity
- Improve model performance using RandomizedSearchCV Hyperparameter Tuning
- Evaluate model using Accuracy, Precision, Recall and F1 Score
- Use Classification Report for detailed class-wise performance analysis
- Compare Before and After tuning performance

### 1. Import Libraries

In [39]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

### 2. Load Dataset

In [40]:
df = pd.read_csv("Adult_Dataset.csv")

### 3. Data Understanding

In [41]:
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [42]:
df.shape

(32561, 15)

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [44]:
df.describe()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


### 4. Data Preprocessing

#### 4.1 Handle Missing Values

In [45]:
df.replace(' ?', np.nan, inplace=True)
print("Missing values after replace:")
print(df.isnull().sum())
df.dropna(inplace=True)

Missing values after replace:
age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64


#### 4.2 Encoding

In [46]:
df = pd.get_dummies(df, drop_first=True, dtype=int)

### 5. Model Building

#### 5.1 Split Features & Target

In [47]:
X = df.drop('income_>50K', axis=1)
y = df['income_>50K']

#### 5.2 Train-Test Split

In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#### 5.3 Model Before Tuning

- ##### Model selection

In [49]:
model_before = RandomForestClassifier(random_state=42)

- ##### Train Model

In [50]:

model_before.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

- ##### Make Predictions

In [51]:

y_pred_before = model_before.predict(X_test)

- ##### Model Evaluation

In [52]:
print("BEFORE TUNING")
print("Accuracy  :", accuracy_score(y_test, y_pred_before))
print("Precision :", precision_score(y_test, y_pred_before))
print("Recall    :", recall_score(y_test, y_pred_before))
print("F1 Score  :", f1_score(y_test, y_pred_before))

BEFORE TUNING
Accuracy  : 0.8518347919545525
Precision : 0.7183206106870229
Recall    : 0.6122316200390371
F1 Score  : 0.6610467158412364


#### 5.4 Hyperparameter Tuning


- ##### Define Hyperparameters

In [53]:
params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}


- ##### Apply RandomizedSearchCV"

In [54]:

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=params,
    n_iter=20,
    cv=3,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)

#### 5.5 Model After Tuning

- ##### Train with RandomSearchCV

In [55]:
random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=3, estimator=RandomForestClassifier(random_state=42),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': [None, 10, 20, 30],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300]},
                   random_state=42, scoring='accuracy')

- ##### Build Best Model

In [56]:
best_model = random_search.best_estimator_

- ##### Make Predictions

In [57]:
y_pred_after = best_model.predict(X_test)

- ##### Model Evaluation

In [58]:
print("AFTER TUNING")
print("Best Parameters :", random_search.best_params_)
print("Accuracy  :", accuracy_score(y_test, y_pred_after))
print("Precision :", precision_score(y_test, y_pred_after))
print("Recall    :", recall_score(y_test, y_pred_after))
print("F1 Score  :", f1_score(y_test, y_pred_after))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_after))

AFTER TUNING
Best Parameters : {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 30, 'criterion': 'entropy'}
Accuracy  : 0.8608935974205435
Precision : 0.7712811693895099
Recall    : 0.5836044242029929
F1 Score  : 0.6644444444444444

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.95      0.91      4976
           1       0.77      0.58      0.66      1537

    accuracy                           0.86      6513
   macro avg       0.83      0.77      0.79      6513
weighted avg       0.85      0.86      0.85      6513



### **Key Insights**
- Accuracy improved from 85.18% → 86.09% after hyperparameter tuning 

- Precision improved from 0.718 → 0.771
  → Model became more reliable with fewer false positives
- Recall slightly decreased from 0.612 → 0.584
  → Due to stricter classification boundaries after tuning
- F1 Score improved from 0.661 → 0.664 showing better overall balance
- Best Parameters found by RandomizedSearchCV:
  criterion=entropy, max_depth=30, n_estimators=100,
  min_samples_leaf=2, min_samples_split=5
- n_estimators=100 → 100 trees combined for stronger prediction
- Random Forest outperformed single Decision Tree (85% → 86.09%)
- Class imbalance present: 4976 (<=50K) vs 1537 (>50K)
  → Minority class (>50K) harder to detect → lower recall
- Classification Report shows weighted avg F1 = 0.85 → strong model 
- RandomizedSearchCV with n_iter=20, cv=3 used for faster tuning
  vs GridSearchCV which took 10+ minutes

### **Conclusion**
The Random Forest Classifier successfully predicted individual income
categories with an accuracy of 86.09% after hyperparameter tuning —
an improvement from the 85.18% baseline.

Compared to a single Decision Tree (~85%), Random Forest delivered
higher accuracy and precision by combining predictions from 100 trees,
reducing overfitting and improving generalization on unseen data.

RandomizedSearchCV efficiently found the best hyperparameters in
significantly less time compared to GridSearchCV, making it a practical
choice for large datasets. The Classification Report confirmed strong
performance with weighted average F1 Score of 0.85.

This project demonstrates the real-world advantage of ensemble learning
over single models for income classification tasks involving class
imbalance and mixed data types.